#36615 - Final Project Part 1 - Data Loading and Cleaning
## Team Polyhymnia
### Vraj Patel, Jay Louissant, Elaine Yin


#### Description:
This notebook loads the airline delay data, cleans it, and saves it to the metastore for future analysis.

In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

#### 1. Load the airlines data and examine schema:

In [0]:
# load the data from the csv into a spark dataframe
delays = spark.read.csv("abfss://sampledata@lsdsampledata2026.dfs.core.windows.net/delays",
                        header=True, nullValue="NULL")

In [0]:
# examine columns and types
delays.printSchema()

We see that all columns are saved as strings, which is as expected because we loaded the data from a csv file which saves all data in text format. Thus, we will have to convert each column into an appropriate type.

In [0]:
# see how many rows are in the dataframe
delays.count()

We are dealing with a large dataset with over 61 million flights over the ten year span. Next, we observe the data in the first several rows to get a sense of the data values.

In [0]:
delays.limit(5).show()

Upon viewing the first few rows of the data, we observe that all the times and intervals of times have been recorded to the nearest minute, and the features representing the minutes of delay by reason are null instead of zero for ontime flights. We also observe an extra "unnamed" column and a "CANCELLATION_CODE" column (contains null values) that are not mentioned in the data dictionary of needed columns.

#### 2. Clean the data by casting columns to appropriate types:

Many columns are numeric or dates but loaded as strings from CSV. Now we convert columns from string to appropriate types.

In [0]:
# custom function to convert time in hhmm format to number of minutes since midnight
def hhmm_to_minutes(hhmm):
    if hhmm is None:
        return None
    return hhmm // 100 * 60 + hhmm % 100

# register the function as a spark UDF (user defined function)
time_udf = F.udf(hhmm_to_minutes, T.IntegerType())


In [0]:
# convert date column
delays = delays.withColumn("FL_DATE", F.to_date(F.col("FL_DATE"), "yyyy-MM-dd"))

# convert time columns (convert hhmm format to minutes since midnight)
time_columns = ["CRS_DEP_TIME", "DEP_TIME", "WHEELS_OFF", "WHEELS_ON", 
                "CRS_ARR_TIME", "ARR_TIME"]
for col_name in time_columns:
    delays = delays.withColumn(col_name, time_udf(F.col(col_name).cast(T.DoubleType()).cast(T.IntegerType())))

# convert delay and duration columns to integers (they're in minutes)
int_columns = ["OP_CARRIER_FL_NUM", "DEP_DELAY", "TAXI_OUT", "TAXI_IN", 
               "ARR_DELAY", "CRS_ELAPSED_TIME", "ACTUAL_ELAPSED_TIME", 
               "AIR_TIME", "DISTANCE"]
for col_name in int_columns:
    delays = delays.withColumn(col_name, F.col(col_name).cast(T.DoubleType()).cast(T.IntegerType()))

# convert boolean flags (CANCELLED, DIVERTED)
delays = delays.withColumn("CANCELLED", F.col("CANCELLED").cast(T.DoubleType()).cast(T.IntegerType()))
delays = delays.withColumn("DIVERTED", F.col("DIVERTED").cast(T.DoubleType()).cast(T.IntegerType()))

# convert delay reason columns to integers
delay_reason_columns = ["CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", 
                        "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY"]
for col_name in delay_reason_columns:
    delays = delays.withColumn(col_name, F.col(col_name).cast(T.DoubleType()).cast(T.IntegerType()))

# keep as strings: OP_CARRIER, ORIGIN, DEST (these are codes)

# drop the last column which is "unnamed", and the "CANCELLATION_CODE" column
delays = delays.drop("Unnamed: 27")
delays = delays.drop("CANCELLATION_CODE")

# verify the new schema
delays.printSchema()


In [0]:
# convert null values to 0, for the delay reasons
delays = delays.fillna(0, subset=delay_reason_columns)


#### 3. Save cleaned data to metastore

In [0]:
# save dataframe as table
table_name = "lsd_2026.default.polyhymnia_cleaned_airline"
delays.write.saveAsTable(table_name)


Now, the "polyhymnia_cleaned_airline" table can be retrieved and analyzed in a separate notebook.